In [1]:
# All paths in this notebook are relative to the repository root; anchor the working directory there
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Peak-Hour Factors on the V2 Routes

Step 20 (`Corridor_peak_hour_2022.ipynb`) estimated peak-hour factors from the survey's minute-level departure times for the trips between the 18 line areas of the earlier aggregation, per direction and layer. Step 24 applied those factors to the V2 routes unchanged. This notebook re-estimates them **on the V2 route sequences themselves** — per route (T1 Nazareth, T2 Krayot, T3 Kiryat Yam), per direction (up = away from Tirat Carmel, down = towards it) and per layer (car; bus; taxi-type) — and on the **tree network** (every pair of the 25 areas on its unique path, each link crossing counted in its own direction). Same construction as step 20: 15-minute bins over 06:00–09:00, trips allocated to area pairs with the population / employment split and weighted by the links they cross; **PHF₃ₕ** = peak-hour share of the three hours, **PHF₆₀** = peak hour ÷ 4 × busiest quarter; 200-replicate household bootstrap; the direction-level factor is applied where the sample has at least 100 sampled trips, otherwise the layer's study-area factor. Rail takes the bus factor.

Step 24 reads the applied factors written here (`Output/corridor_v2/peak_hour_factors_v2_applied.csv`) when the file exists, so the run order is this notebook first, then `Corridor_flow_profile_V2_routes.ipynb`.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/corridor_v2'; os.makedirs(OUT, exist_ok=True); os.makedirs('Output/figures', exist_ok=True)
def style_ax(ax, title=None, xlabel=None, ylabel=None):
    ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=9)
    if title: ax.set_title(title, color=INK, fontsize=11)
    if xlabel: ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=INK2, fontsize=10)

# ---- survey trips with minute-level departure time (same extraction as steps 15 / 20) ----
df = pd.read_excel('Input/THS_2017-2018/trips_ths_2017.xlsx')
d = df.sort_values(['PerID3', 'SurveyDay', 'placeno']).copy(); g = d.groupby(['PerID3', 'SurveyDay'])
d['origin'] = g['actTaz'].shift(1); d['trip_dep_h'] = g['Dep_h'].shift(1); d['dep'] = g['STDep'].shift(1)
trips = d.dropna(subset=['origin', 'actTaz', 'trip_dep_h']); trips = trips[trips['trip_dep_h'].isin([6, 7, 8])].copy()
NB = 12; trips['bin'] = ((trips['dep'] - 6.0) * 4).astype(int).clip(0, NB - 1)
COMP = {'CAR': [10, 11], 'BUS': [3, 5], 'TAXI': [4, 8], 'RAIL': [7]}   # codes confirmed against the activities file, step 33 (23 Sep 2026): 3 Public Bus, 5 Matronit; 4 group taxi, 8 special taxi
trips['comp'] = trips['mainmode'].map(lambda m: next((k for k, v in COMP.items() if m in v), 'OTHER'))
k26 = pd.read_excel('Input/TAZ_2636_Keys.xlsx').drop_duplicates('TAZ_2636'); map_2636_1250 = k26.set_index('TAZ_2636')['TAZ_1250']
trips['o1250'] = trips['origin'].map(map_2636_1250); trips['d1250'] = trips['actTaz'].map(map_2636_1250)
KEYS_LFS = 'Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv'
keys_raw = pd.read_csv('Input/taz_keys_from_shapefile.csv') if open(KEYS_LFS, 'rb').read(40).startswith(b'version https://git-lfs') else pd.read_csv(KEYS_LFS, encoding='windows-1255')
kd = keys_raw[['TAZ_1270', 'TAZ_NUMBER', 'SZ_NEW']].dropna(subset=['TAZ_NUMBER']).astype({'TAZ_NUMBER': int, 'SZ_NEW': int}).drop_duplicates('TAZ_NUMBER')
TAZ = np.array(sorted(kd['TAZ_NUMBER'])); N = len(TAZ); taz_pos = {t: i for i, t in enumerate(TAZ)}
children = kd.groupby('TAZ_1270')['TAZ_NUMBER'].apply(list); north_1250 = sorted(children.index); z_idx = {z: i for i, z in enumerate(north_1250)}
tm = trips[trips['o1250'].isin(z_idx) & trips['d1250'].isin(z_idx)].copy(); tm['oi'] = tm['o1250'].map(z_idx); tm['di'] = tm['d1250'].map(z_idx)
zon = pd.read_csv('Input/Zonal_2020.csv', encoding='windows-1255').set_index('TAZ_ID').reindex(TAZ)
pop, emp = zon['POPULATION'].fillna(0).values, zon['EMPL_TOT'].fillna(0).values
def alloc(primary, secondary):
    S = np.zeros((len(north_1250), N))
    for z, kids in children.items():
        idx = [taz_pos[t] for t in kids]
        for v in (primary[idx], secondary[idx], np.ones(len(idx))):
            if v.sum() > 0: S[z_idx[z], idx] = v / v.sum(); break
    return S
S_o, S_d = alloc(pop, emp), alloc(emp, pop)
# ---- V2 areas, routes and the tree ----
xl = pd.ExcelFile('Input/Corridor_TAZ_Agg_V2.xlsx'); areas = xl.parse('AreaCodes').set_index('AggCode'); key = xl.parse('TazAgg')
area_of = key.set_index('TAZ')['AggCode']; AREAS = list(areas.index); apos = {a: i for i, a in enumerate(AREAS)}; names = areas['AggAreaName']
ROUTES = {r: list(areas[areas[f'Order_{r}'] > 0].sort_values(f'Order_{r}').index) for r in ['T1', 'T2', 'T3']}
TRUNK = [a for a in ROUTES['T1'] if all(a in ROUTES[r] for r in ROUTES)]; BRANCH = {r: [a for a in seq if a not in TRUNK] for r, seq in ROUTES.items()}
M_area = np.zeros((N, len(AREAS)))
for i, t in enumerate(TAZ):
    if t in area_of.index: M_area[i, apos[area_of[t]]] = 1
SA_o, SA_d = S_o @ M_area, S_d @ M_area
# tree path: number of links crossed in each direction between two areas
def to_root(a):
    if a in TRUNK: return list(reversed(TRUNK[:TRUNK.index(a) + 1]))
    r = next(r for r in ROUTES if a in BRANCH[r]); i = BRANCH[r].index(a)
    return list(reversed(BRANCH[r][:i + 1])) + list(reversed(TRUNK))
def dir_links(o, d_):
    if o == d_: return 0, 0
    po, pd_ = to_root(o), to_root(d_); common = next(a for a in po if a in set(pd_))
    return pd_.index(common), po.index(common)          # (links up, links down)
UPL = np.array([[dir_links(o, d_)[0] for d_ in AREAS] for o in AREAS]); DNL = np.array([[dir_links(o, d_)[1] for d_ in AREAS] for o in AREAS])
print(f"AM trips in the study area: {len(tm):,} sampled; V2: {len(AREAS)} areas, routes " + ', '.join(f'{r} {len(s)} areas' for r, s in ROUTES.items()))

AM trips in the study area: 26,863 sampled; V2: 25 areas, routes T1 17 areas, T2 16 areas, T3 16 areas


## 1. Departure-time profiles per route, direction and layer — and on the tree network

In [3]:
def pair_table(sub):
    """Long table: sampled trip row, origin area, destination area, allocation share (product of the two shares)."""
    So, Sd = SA_o[sub['oi'].values], SA_d[sub['di'].values]; rows = []
    for r_, (so, sd) in enumerate(zip(So, Sd)):
        nzo, nzd = np.nonzero(so)[0], np.nonzero(sd)[0]
        for co in nzo:
            for cd in nzd:
                if co != cd: rows.append((r_, co, cd, so[co] * sd[cd]))
    return pd.DataFrame(rows, columns=['row', 'o', 'd', 'share'])
def window_stats(s):
    s = np.asarray(s, float); tot = s.sum()
    if tot <= 0: return dict(PHF3h=np.nan, start=np.nan, PHF60=np.nan)
    p = s / tot; win = np.array([p[i:i + 4].sum() for i in range(NB - 3)]); k = int(np.argmax(win))
    return dict(PHF3h=win[k], start=6 + k / 4, PHF60=win[k] / (4 * p[k:k + 4].max()))
LAYERS = ['CAR', 'BUS', 'TAXI']; rng = np.random.default_rng(7); B = 200
prof = {}; rows = []
for comp in LAYERS:
    sub = tm[tm['comp'] == comp].reset_index(drop=True); pt = pair_table(sub)
    pt['wf'] = sub['new_wf'].values[pt['row']]; pt['bin'] = sub['bin'].values[pt['row']]; pt['hh'] = sub['HHID3'].values[pt['row']]
    for scope in ['T1', 'T2', 'T3', 'network']:
        seq = ROUTES.get(scope); t = pt.copy()
        if seq is not None:
            spos = {apos[a]: i for i, a in enumerate(seq)}; t = t[t['o'].isin(spos) & t['d'].isin(spos)].copy()
            t['up'] = np.clip(t['d'].map(spos) - t['o'].map(spos), 0, None); t['down'] = np.clip(t['o'].map(spos) - t['d'].map(spos), 0, None)
        else:
            t['up'] = UPL[t['o'].values, t['d'].values]; t['down'] = DNL[t['o'].values, t['d'].values]
        for dname in ['up', 'down']:
            td = t[t[dname] > 0].copy(); td['w'] = td['share'] * td['wf'] * td[dname]           # link-crossing weight in this direction
            prof[(comp, scope, dname)] = td
            s = np.bincount(td['bin'].values, weights=td['w'].values, minlength=NB); st = window_stats(s)
            hh_ids = np.unique(td['hh'].values); hpos = {h: i for i, h in enumerate(hh_ids)}; hidx = np.array([hpos[h] for h in td['hh'].values], dtype=int); b3, b60 = [], []   # dtype fixed 23 Sep 2026: a layer-direction with no sampled trips (taxi-type under the corrected codes) gave an empty float index
            for _ in range(B):
                mult = np.bincount(rng.integers(0, len(hh_ids), len(hh_ids)), minlength=len(hh_ids))
                sb = np.bincount(td['bin'].values, weights=td['w'].values * mult[hidx], minlength=NB); sbst = window_stats(sb); b3.append(sbst['PHF3h']); b60.append(sbst['PHF60'])
            rows.append({'scope': scope, 'layer': comp, 'direction': dname, 'n sampled': int(td['row'].nunique()), **st, 'PHF3h p5': np.nanpercentile(b3, 5), 'PHF3h p95': np.nanpercentile(b3, 95), 'PHF60 p5': np.nanpercentile(b60, 5), 'PHF60 p95': np.nanpercentile(b60, 95)})
for comp in LAYERS + ['RAIL']:
    sub = tm[tm['comp'] == comp]; st = window_stats(sub.groupby('bin')['new_wf'].sum().reindex(range(NB)).fillna(0).values)
    rows.append({'scope': 'study area', 'layer': comp, 'direction': '—', 'n sampled': len(sub), **st})
phf = pd.DataFrame(rows)
phf['peak hour'] = phf['start'].map(lambda s: f"{int(s):02d}:{int((s % 1) * 60):02d}–{int(s + 1):02d}:{int((s % 1) * 60):02d}" if pd.notna(s) else '')
phf.to_csv(f'{OUT}/peak_hour_factors_v2.csv', index=False, float_format='%.4f')
print(phf[['scope', 'layer', 'direction', 'n sampled', 'peak hour', 'PHF3h', 'PHF3h p5', 'PHF3h p95', 'PHF60']].round(3).to_string(index=False))

     scope layer direction  n sampled   peak hour  PHF3h  PHF3h p5  PHF3h p95  PHF60
        T1   CAR        up        203 07:15–08:15  0.626     0.500      0.756  0.756
        T1   CAR      down        240 07:00–08:00  0.578     0.441      0.719  0.758
        T2   CAR        up        332 07:15–08:15  0.619     0.505      0.758  0.806
        T2   CAR      down        447 07:15–08:15  0.569     0.510      0.688  0.589
        T3   CAR        up        241 07:15–08:15  0.652     0.534      0.790  0.782
        T3   CAR      down        299 07:00–08:00  0.569     0.478      0.672  0.646
   network   CAR        up        777 07:15–08:15  0.556     0.493      0.636  0.789
   network   CAR      down       1001 07:00–08:00  0.522     0.446      0.609  0.885
        T1   BUS        up         74 06:45–07:45  0.740     0.618      0.879  0.527
        T1   BUS      down         37 06:15–07:15  0.496     0.432      0.736  0.580
        T2   BUS        up        103 07:30–08:30  0.592     0.54

/usr/local/lib/python3.11/dist-packages/numpy/lib/_nanfunctions_impl.py:1396: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/usr/local/lib/python3.11/dist-packages/numpy/lib/_nanfunctions_impl.py:1396: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


## 2. Which factor is applied, per route / network, layer and direction

In [4]:
N_MIN_DIR = 100
areafac = phf[phf['scope'] == 'study area'].set_index('layer')['PHF3h']
old = pd.read_csv('Output/ths2017/three_mode_2022/peak_hour_factors_applied.csv'); old['direction'] = old['direction'].map({'1→23': 'up', '23→1': 'down'})
oldfac = old.set_index(['layer', 'direction'])['PHF3h applied']
app = []
for scope in ['T1', 'T2', 'T3', 'network']:
    for comp in LAYERS:
        for dname in ['up', 'down']:
            r = phf[(phf['scope'] == scope) & (phf['layer'] == comp) & (phf['direction'] == dname)].iloc[0]
            use_dir = r['n sampled'] >= N_MIN_DIR
            app.append({'scope': scope, 'layer': comp.lower(), 'direction': dname, 'PHF3h applied': r['PHF3h'] if use_dir else areafac[comp],
                        'basis': f"{scope}, direction ({int(r['n sampled'])} sampled)" if use_dir else f"study area, all trips ({int(phf[(phf.scope == 'study area') & (phf.layer == comp)]['n sampled'].iloc[0])} sampled)",
                        'direction value': r['PHF3h'], 'direction p5': r['PHF3h p5'], 'direction p95': r['PHF3h p95'], 'study-area value': areafac[comp], 'step-20 value (18-area line)': oldfac[(comp, dname)]})
        app.append({'scope': scope, 'layer': 'rail', 'direction': 'up', 'PHF3h applied': app[-2]['PHF3h applied'], 'basis': 'bus factor', 'step-20 value (18-area line)': oldfac[('BUS', 'up')]})
        app.append({'scope': scope, 'layer': 'rail', 'direction': 'down', 'PHF3h applied': app[-2]['PHF3h applied'], 'basis': 'bus factor', 'step-20 value (18-area line)': oldfac[('BUS', 'down')]})
applied = pd.DataFrame(app); applied.to_csv(f'{OUT}/peak_hour_factors_v2_applied.csv', index=False, float_format='%.4f')
print(applied[applied.layer != 'rail'][['scope', 'layer', 'direction', 'PHF3h applied', 'basis', 'direction value', 'direction p5', 'direction p95', 'step-20 value (18-area line)']].round(3).to_string(index=False))

  scope layer direction  PHF3h applied                                basis  direction value  direction p5  direction p95  step-20 value (18-area line)
     T1   car        up          0.626          T1, direction (203 sampled)            0.626         0.500          0.756                         0.661
     T1   car      down          0.578          T1, direction (240 sampled)            0.578         0.441          0.719                         0.626
     T1   bus        up          0.589 study area, all trips (1790 sampled)            0.740         0.618          0.879                         0.589
     T1   bus      down          0.589 study area, all trips (1790 sampled)            0.496         0.432          0.736                         0.589
     T1  taxi        up          0.588   study area, all trips (99 sampled)            1.000         1.000          1.000                         0.588
     T1  taxi      down          0.588   study area, all trips (99 sampled)            0

In [5]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8.5), facecolor='white', sharey=True)
x15 = 6 + np.arange(NB) / 4
for j, scope in enumerate(['T1', 'T2', 'T3', 'network']):
    for i, dname in enumerate(['up', 'down']):
        ax = axes[i, j]
        for comp, color in [('CAR', MUTED), ('BUS', BLUE), ('TAXI', AQUA)]:
            td = prof[(comp, scope, dname)]; s = np.bincount(td['bin'].values, weights=td['w'].values, minlength=NB); p = s / s.sum() if s.sum() > 0 else s
            n = int(td['row'].nunique()); used = n >= N_MIN_DIR
            ax.stairs(p, np.append(x15, 9.0), color=color, linewidth=2.2 if used else 1.1, alpha=1 if used else 0.6, label=f"{comp.lower() if comp != 'TAXI' else 'taxi-type'} ({n} sampled){' — applied' if used else ''}")
            if not used:
                sa = tm[tm['comp'] == comp].groupby('bin')['new_wf'].sum().reindex(range(NB)).fillna(0).values
                ax.stairs(sa / sa.sum(), np.append(x15, 9.0), color=color, linewidth=2.2, linestyle='--', label=f"{comp.lower() if comp != 'TAXI' else 'taxi-type'}, study area — applied ({areafac[comp]:.2f})")
        ax.axvspan(7, 8, color=BLUE, alpha=0.06, lw=0)
        ax.set_xticks(np.arange(6, 9.01, 0.5)); ax.set_xticklabels([f'{int(v):02d}:{int((v % 1) * 60):02d}' for v in np.arange(6, 9.01, 0.5)], fontsize=7); ax.set_xlim(6, 9)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
        style_ax(ax, f'{scope} — {dname} ({"away from" if dname == "up" else "towards"} Tirat Carmel)', 'departure time' if i == 1 else None, 'share of 3-hour link crossings per 15 min' if j == 0 else None); ax.legend(frameon=False, fontsize=7)
fig.suptitle('Departure-time profiles of the trips between V2 areas, weighted by links crossed — per route and on the tree network (shaded: 07:00–08:00)', color=INK, fontsize=12, y=1.0)
plt.tight_layout(); fig.savefig('Output/figures/corridor_v2_peak_hour_profiles.png', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()

## Findings

- **Car factors are identifiable on every V2 route and direction** (203–447 sampled trips each; 777 / 1,001 on the tree network). Up (away from Tirat Carmel): T1 0.626, T2 0.619, T3 0.652, peak hour 07:15–08:15 on all three. Down (towards Tirat Carmel, i.e. into Haifa from the east and north): 0.578 / 0.569 / 0.569, peak hour 07:00–08:00 (T2 07:15–08:15). On the tree network — which adds the branch-to-branch pairs and pools the three routes — 0.556 up and 0.522 down: the long trips from Nazareth and the Krayot start earlier and spread their departures over 06:00–08:00, so the pooled down profile is flatter than any single route's.
- **Against step 20** (0.661 up / 0.626 down on the 18-area line): the up factors are 1–6 % lower and the down factors 8–9 % lower per route, 16–17 % lower on the network. The step-24 first pass, which applied the step-20 factors unchanged, over-stated the peak-hour car and total flows by those margins; step 24 now reads this notebook's factors.
- **Bus and taxi-type stay on the study-area factors** (0.590 / 0.580): the largest route-direction bus sample is 58 trips (96 on the network up direction), below the 100-trip rule, and the route values scatter 0.43–0.69 with bootstrap ranges of 0.43–0.84 around 0.59. Taxi-type route values (0.58–0.91 on 17–48 trips) all sit above the study-area 0.58, so the applied factor may under-state the taxi-type peak on the corridor; with these samples it cannot be pinned down.
- **Same caveats as step 20.** Departure-hour factors, not link-crossing; household-bootstrap ranges of ± 0.1 on car; rail takes the bus factor; a boarding-hour factor from the RavKav files (LFS) is still the independent check for bus (task B1e).
- **Outputs.** `Output/corridor_v2/peak_hour_factors_v2.csv` (every scope × layer × direction with hourly windows and bootstrap ranges), `peak_hour_factors_v2_applied.csv` (the factor step 24 applies, its basis, and the step-20 value alongside); figure `corridor_v2_peak_hour_profiles.png`.